# Spark Vs. Hadoop


## shuffle对比


Shuffle描述着数据从`map task`输出到`reduce task`输入的这段过程。

shuffle是连接Map和Reduce之间的桥梁，Map的输出要用到Reduce中必须经过shuffle这个环节，shuffle的性能高低直接影响了整个程序的性能和吞吐量。

因为在分布式情况下，reduce task需要跨节点去拉取其它节点上的map task结果。这一过程将会产生网络资源消耗和内存，磁盘IO的消耗。

通常shuffle分为两部分：
- Map阶段的数据准备: map端的Shuffle称之为`Shuffle Write`
- Reduce阶段的数据拷贝处理: Reduce端的Shuffle称之为`Shuffle Read`

### Hadoop Shuffle

Hadoop的计算模式 Map-Reudce

- 提供数据端: Mapper - 每个生成数据的任务
- 接收数据端: Reducer - 每个拉取数据的任务

Shuffle的本质是: 

将Map端获得的数据使用`分区器`(Partitioner)进行划分,并将数据发送给对应的`Reducer`

具体细节:  

(1) Map端

Map任务运行,每一个 map task 都有一个内存缓冲区, 存储map的输出结果

缓冲区数据满的时候,缓冲区的数据以临时文件的方式写入磁盘

整个Map task执行结束之后,对磁盘中对应于此map task的临时文件进行合并.

合并成一个正式完整的文件,等待reduce task拉取数据

(2) Reducer端

执行之前需要不断地拉取当前job里每一个map task的最终结果,然后合并(merge)不同地方的数据,形成一个文件,作为reduce task的输入文件

(3) shuffle衔接

mapper运行后,通过 partitioner 决定当前的map输出应该交由哪个 reduce task 运行.

### Spark Shuffle

spark的计算模式是基于`stage`的`pipline`

shuffle就存在于stage之间

stage划分的主要依据是`宽依赖`

spark中负责shuffle过程执行,计算和处理的组件主要是ShuffleManager(Shuffle管理器),
stage之间拉取数据时,会根据索引读取每个磁盘文件中的部分数据.

ShuffleManager有两种实现方式:
- Hash shuffle 
- Sort Shuffle

spark的 shuffle 类算子(groupByKey,combineByKey,reduceByKey)

每个task处理的数据按 key 进行"分区", 对key进行hash算法,将结果相同的item放入一个磁盘文件中, 每一个磁盘文件属于reduce端stage里的一个task

关于数量的问题,shuffle write task需要为下一阶段的stage创建多少磁盘文件?

**下一个stage的task有多少个**,当前stage的每一个task就需要创建多少份磁盘文件

**shuffle read的准备阶段:**

shuffle的每一个task需要将计算节点中所有相同key的数据,从节点,通过网络拉取到本地节点上

按照key进行聚合或者链接...

每一个shuffle read task会有一个自己的缓冲区(buffer),每次只能拉取缓冲区相同大小的数据,然后通过内存中的map进行聚合操作,**边拉边操作**

上面讲述的 hash shuffle,可以看出其主要问题有:
- 产生大量小文件
- IO占用时间多
- 可能会导致 OOM 

### Hash shuffle的合并机制

使用 HashShuffleManager 时建议开启此操作

当 本地executor执行多次同样 stage task 时, 能够通过合并等位置文件起到优化作用,从而优化IO,网络时间等.

此时引入了 shuffleFileGroup (文件组)

### Sort Shuffle

有两种运行机制:

- 普通运行机制
- bypass 运行机制

## 总结
Shuffle 过程本质上都是将 Map 端获得的数据使用分区器进行划分，并将数据发送给对应的 Reducer 的过程。

shuffle作为处理连接map端和reduce端的枢纽，其shuffle的性能高低直接影响了整个程序的性能和吞吐量。map端的shuffle一般为shuffle的Write阶段，reduce端的shuffle一般为shuffle的read阶段。Hadoop和spark的shuffle在实现上面存在很大的不同，spark的shuffle分为两种实现，分别为HashShuffle和SortShuffle，

HashShuffle又分为普通机制和合并机制，普通机制因为其会产生M*R个数的巨量磁盘小文件而产生大量性能低下的Io操作，从而性能较低，因为其巨量的磁盘小文件还可能导致OOM，HashShuffle的合并机制通过重复利用buffer从而将磁盘小文件的数量降低到Core*R个，但是当Reducer 端的并行任务或者是数据分片过多的时候，依然会产生大量的磁盘小文件。

SortShuffle也分为普通机制和bypass机制，普通机制在内存数据结构(默认为5M)完成排序，会产生2M个磁盘小文件。而当shuffle map task数量小于spark.shuffle.sort.bypassMergeThreshold参数的值。或者算子不是聚合类的shuffle算子(比如reduceByKey)的时候会触发SortShuffle的bypass机制，SortShuffle的bypass机制不会进行排序，极大的提高了其性能

## spark 和 hadoop 的对比


- mapreduce 读 – 处理 - 写磁盘 -- 读 - 处理 - 写
- spark     读 - 处理 - 处理   --（`需要的时候`）写磁盘 - 写

`需要的时候`是指:shuffle类算子

问题一: `需要的时候`是指?

问题二: spark的优势?

1. 中间结果进内存,迭代效率更高,延迟加载提高了处理效率,DAG式的分布式并行计算编程框架
2. 基于RDD的分布式数据集抽象提供了更高的容错
3. 更加通用的大数据处理模式,更加丰富的大数据分析处理生态

问题三: shuffle的区别?

1. spark是DAG类型的数据流处理,所以spark可以同时处理map stages tasks,但是Hadoop只能`读-处理-写-读...`
2. 实现方式的不一样,spark更加丰富的shuffle机制,具包括:sort和hash,MR基本只有sort类型,着取决于MapReduce的设计之初就是用来处理键值对
3. mr会进行两次排序,spark提供了选择,因为spark的处理能力更强,所以没有默认两次排序
4. 效率差异
5. MR是粗粒度(满了才处理)的数据获取,Spark是细粒度的数据获取(存在相同key就处理)